In [ ]:
# hide
from typing import Iterator
import numpy as np
import pyquist as pq

In [ ]:
# hide
def iter_frames(audio: pq.Audio, N_H: int, N_F: int) -> Iterator[np.ndarray]:
    for start in range(0, len(audio) - N_F + 1, N_H):
        yield audio.samples[start:start + N_F]


def rms(x: np.ndarray) -> float:
    """The root-mean-square amplitude of x."""
    return np.sqrt(np.mean(x ** 2))


def rms_envelope(audio: pq.Audio, N_H: int, N_F: int) -> np.ndarray:
    """The RMS amplitude of each frame of the audio (Chapter 10)."""
    return np.array([rms(frame) for frame in iter_frames(audio, N_H, N_F)])


def compress(audio: pq.Audio, threshold_db: float, ratio: float,
             key: pq.Audio | None = None) -> pq.Audio:
    """Turn the audio down wherever the level of `key` (the audio itself, by
    default) exceeds the threshold, so the level above the threshold shrinks by `ratio`."""
    key = audio if key is None else key
    N_H = key.sample_rate // 100                         # 100 frames per second
    N_F = 4 * N_H
    level_db = pq.helper.amplitude_to_db(np.maximum(rms_envelope(key, N_H, N_F), 1e-6))
    target_db = np.where(level_db > threshold_db,
                         threshold_db + (level_db - threshold_db) / ratio, level_db)
    gain = pq.helper.db_to_amplitude(target_db - level_db)   # one gain per frame...
    centers = N_H * np.arange(len(gain)) + N_F // 2         # ...at the center of each frame...
    gain = np.interp(np.arange(len(audio)), centers, gain)  # ...interpolated to every sample
    return pq.Audio(audio.samples * gain[:, np.newaxis], audio.sample_rate)

In [ ]:
pad = pq.Audio.from_file("../assets/pad.wav")
kick = pq.Audio.from_file("../assets/kick-loop.wav")
N = min(len(pad), len(kick))
pad, kick = 0.5 * pad[:N], kick[:N]
ducked = compress(pad, threshold_db=-40.0, ratio=10.0, key=kick)   # the kick turns the pad down
pq.play(pad + kick, normalize=True)
pq.play(ducked + kick, normalize=True)